In [1]:
# Cập nhật danh sách gói (bắt buộc)
!sudo apt-get update

# Cài đặt công cụ zstd để giải nén file .tar.zst của Ollama phiên bản mới
!sudo apt-get install -y zstd

# Cài đặt các thư viện Python
%pip install ollama

!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [356 B]       
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]  
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 

In [3]:
import subprocess
import time

# (Tùy chọn) Kill tiến trình ollama cũ nếu có, để tránh xung đột cổng
subprocess.run("pkill ollama", shell=True, stderr=subprocess.DEVNULL)
time.sleep(1)

# Khởi động ollama serve trong background
# stdout và stderr được chuyển hướng để tránh làm lộn xộn output của notebook
subprocess.Popen("ollama serve", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Đợi vài giây cho server khởi động hoàn toàn
time.sleep(4)
print("✅ Ollama server đã được khởi động thành công!")

✅ Ollama server đã được khởi động thành công!


In [4]:
!ollama pull qwen2.5:7b-instruct

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 2bada8a74506:   0% ▕                  ▏ 8.4 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   2% ▕                  ▏  90 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   3% ▕                  ▏ 141 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   5% ▕                  ▏ 250 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   7% ▕█                 ▏ 348 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   9% ▕█                 ▏ 400 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  11% ▕█                 ▏ 500 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  13% ▕██                ▏ 598 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  14% ▕██                

In [25]:
def build_prompt(problem: str):
    return f"""
You are a physics/math problem classifier.

Your task:
Given a problem statement, classify it into EXACTLY ONE of the following categories:

- coordinates
- circuit
- equations
- scalar_formula
- theorem_reasoning
- unknown

Definitions:

1. coordinates
Problems involving:
- geometry
- coordinates
- vectors
- electric charges placed at points
- distances between points
- triangles, squares, circles
- electric field / force using positions

2. circuit
Problems involving:
- electric circuits
- resistors
- capacitors
- inductors
- current
- voltage
- Kirchhoff laws
- RLC circuits

3. equations
Problems primarily focused on:
- solving equations
- systems of equations
- algebraic manipulation
- differential equations
- finding unknown variables

4. scalar_formula
Problems solvable mainly by direct formula substitution.
Usually:
- plug known values into formulas
- no deep geometry or reasoning

5. theorem_reasoning
Problems requiring:
- none explicit value
- proof
- derivation
- explanation
- logical reasoning
- theorem application

6. unknown
If none fit clearly.

IMPORTANT RULES:
- Output ONLY the category name.
- Do NOT explain.
- Do NOT output JSON.
- Do NOT output extra text.
- Choose the MOST dominant category.

Examples:

Problem:
"Two point charges, q1 and q2, of equal magnitude and the same sign, are placed in air a distance r apart. A third point charge, q3, is placed at the midpoint of the line segment connecting q1 and q2. What is the net force acting on charge q3?"
Output:
theorem

Problem:
"There are two charges, q1 = 2×10^-6 C and q2 = -2×10^-6 C, placed at points A and B respectively in a vacuum, separated by a distance of 6 cm. A third charge, q3 = 4×10^-6 C, is placed on the perpendicular bisector of AB, at a distance of 4 cm from the line segment AB. What is the magnitude of the net electric force exerted by charges q1 and q2 on charge q3?"
Output:
coordinates

Problem:
"Two electric forces have magnitudes of 6 N and 8 N, acting at an angle of 120° to each other. Calculate the resultant force of these two forces."
Output:
scalar_formula

Problem:
"Two charges separated by 15 cm exert a force of 4.8 N. Given that q1 = q2 = q, find q."
Output:
equations

Problem:
"Prove that the electric field inside a conductor is zero."
Output:
theorem_reasoning

Now classify this problem:

{problem}
"""

In [159]:
from ollama import generate 
def classify(prompt : str):
    response = generate(
        model= "qwen2.5:7b-instruct",
        prompt=build_prompt(prompt),
        options={
            'temparature':0,
            'keep_alive':'60m',
            'num_predict':4000
        }
    )
    return response['response']

q = "Two electric forces, each with a magnitude of 5 N, act at an angle of 60° to each other. What is the resultant force?"
print(classify(q))

scalar_formula


In [38]:
from dataclasses import dataclass, field
from typing import Optional, List, Union
from sympy import symbols


# =========================================================
# REGISTRY
# =========================================================

PRIMITIVES = {
    "Point",
    "Line",
    "Vector",
    "Distance",
    "Angle",
    "Triangle",
    "Circle",
}

CONSTRAINTS = {
    "Parallel",
    "Perpendicular",
    "Collinear",
    "Midpoint",
    "EqualDistance",
    "EqualAngle",
}

RELATIONS = {
    "Inside",
    "Intersect",
    "Tangent",
}


# =========================================================
# BASE
# =========================================================

class GeometryObject:
    pass


class Constraint:
    pass


class Relation:
    pass


# =========================================================
# PRIMITIVES
# =========================================================

@dataclass
class Point(GeometryObject):
    name: str

    x: object = field(default=None)
    y: object = field(default=None)

    def __post_init__(self):
        if self.x is None:
            self.x = symbols(f"{self.name}_x")

        if self.y is None:
            self.y = symbols(f"{self.name}_y")


@dataclass
class Line(GeometryObject):
    A: Point
    B: Point


@dataclass
class Vector(GeometryObject):
    A: Point
    B: Point


@dataclass
class Distance(GeometryObject):
    A: Point
    B: Point

    value: Optional[float] = None
    unit: str = "m"


@dataclass
class Angle(GeometryObject):
    A: Point
    B: Point
    C: Point

    value: Optional[float] = None
    unit: str = "deg"


@dataclass
class Triangle(GeometryObject):
    A: Point
    B: Point
    C: Point


@dataclass
class Circle(GeometryObject):
    center: Point
    radius: Optional[float] = None


# =========================================================
# CONSTRAINTS
# =========================================================

@dataclass
class Parallel(Constraint):
    line1: Line
    line2: Line


@dataclass
class Perpendicular(Constraint):
    line1: Line
    line2: Line


@dataclass
class Collinear(Constraint):
    points: List[Point]


@dataclass
class Midpoint(Constraint):
    midpoint: Point
    A: Point
    B: Point


@dataclass
class EqualDistance(Constraint):
    d1: Distance
    d2: Distance


@dataclass
class EqualAngle(Constraint):
    angle1: Angle
    angle2: Angle


# =========================================================
# RELATIONS
# =========================================================

@dataclass
class Inside(Relation):
    obj: GeometryObject
    container: GeometryObject


@dataclass
class Intersect(Relation):
    obj1: GeometryObject
    obj2: GeometryObject


@dataclass
class Tangent(Relation):
    line: Line
    circle: Circle




In [ ]:
EXAMPLE_DATASET = [

    {
        "problem":
            "Two charges, q1 = 6 × 10^-8 C and "
            "q2 = -6 × 10^-8 C, are placed at "
            "points A and B in air, 8 cm apart. "
            "A third charge, q3 = 6 × 10^-8 C, "
            "is placed at point C, with "
            "CA = 5 cm and CB = 3 cm. "
            "Determine the force acting on q3.",

        "tags": [
            "Point",
            "Distance"
        ],

        "schema": {

            "entities": {

                "points": [

                    {"id": "A"},
                    {"id": "B"},
                    {"id": "C"}
                ],

                "charges": [

                    {
                        "id": "q1",
                        "value": 6e-8,
                        "unit": "C",
                        "at": "A"
                    },

                    {
                        "id": "q2",
                        "value": -6e-8,
                        "unit": "C",
                        "at": "B"
                    },

                    {
                        "id": "q3",
                        "value": 6e-8,
                        "unit": "C",
                        "at": "C"
                    }
                ]
            },
            "geometry":[],
            "relations":[],
            "constraints": [

                {
                    "type": "Distance",
                    "pair": ["A", "B"],
                    "value": 8,
                    "unit": "cm"
                },

                {
                    "type": "Distance",
                    "pair": ["C", "A"],
                    "value": 5,
                    "unit": "cm"
                },

                {
                    "type": "Distance",
                    "pair": ["C", "B"],
                    "value": 3,
                    "unit": "cm"
                }
            ],

            "queries": [

                {
                    "type": "ElectrostaticForce",
                    "target": "q3"
                }
            ]
        }
    },
    {
    "problem":
        "Three charges: q1 = +3 μC, q2 = -2 μC, "
        "and q3 = +1 μC are placed 10 cm apart "
        "on a straight line. Calculate the force "
        "acting on q2.",

    "tags": [
        "Point",
        "Collinear",
        "Distance"
    ],

    "schema": {

        "entities": {

            "points": [

                {"id": "A"},
                {"id": "B"},
                {"id": "C"}
            ],

            "charges": [

                {
                    "id": "q1",
                    "value": 3e-6,
                    "unit": "C",
                    "at": "A"
                },

                {
                    "id": "q2",
                    "value": -2e-6,
                    "unit": "C",
                    "at": "B"
                },

                {
                    "id": "q3",
                    "value": 1e-6,
                    "unit": "C",
                    "at": "C"
                }
            ]
        },

        "geometry": [

            {
                "primitive": "Collinear",

                "points": [
                    "A",
                    "B",
                    "C"
                ]
            }
        ],

        "constraints": [

            {
                "type": "Distance",

                "pair": [
                    "A",
                    "B"
                ],

                "value": 10,

                "unit": "cm"
            },

            {
                "type": "Distance",

                "pair": [
                    "B",
                    "C"
                ],

                "value": 10,

                "unit": "cm"
            }
        ],

        "queries": [

            {
                "type": "ElectrostaticForce",

                "target": "q2"
            }
        ]
    }
},
{
    "problem":
        "A charge q = -1 μC is attracted by two +1 μC charges. "
        "These two positive charges are located on opposite sides "
        "of q, along the same straight line passing through q, "
        "at distances of 5 cm and 12 cm respectively from q.",

    "tags": [
        "Point",
        "Collinear",
        "Distance"
    ],

    "schema": {

        "entities": {

            "points": [

                {"id": "A"},
                {"id": "Q"},
                {"id": "B"}
            ],

            "charges": [

                {
                    "id": "q1",
                    "value": 1e-6,
                    "unit": "C",
                    "at": "A"
                },

                {
                    "id": "q",
                    "value": -1e-6,
                    "unit": "C",
                    "at": "Q"
                },

                {
                    "id": "q2",
                    "value": 1e-6,
                    "unit": "C",
                    "at": "B"
                }
            ]
        },

        "geometry": [

            {
                "primitive": "Collinear",

                "points": [
                    "A",
                    "Q",
                    "B"
                ]
            }
        ],

        "constraints": [

            {
                "type": "Distance",

                "pair": [
                    "A",
                    "Q"
                ],

                "value": 5,

                "unit": "cm"
            },

            {
                "type": "Distance",

                "pair": [
                    "Q",
                    "B"
                ],

                "value": 12,

                "unit": "cm"
            }
        ],

        "queries": [

            {
                "type": "ElectrostaticForce",

                "target": "q"
            }
        ]
    }
},
######TRIANGLE

    {
    "problem":
        "Three electric charges, q1 = q2 = q3 = 1.6 × 10^-19 C, "
        "are placed at the three vertices of an equilateral triangle "
        "ABC with side length 16 cm in air. "
        "Determine the net electric force vector acting on q3.",

    "tags": [
        "Point",
        "EquilateralTriangle"
    ],

    "schema": {

        "entities": {

            "points": [

                {"id": "A"},
                {"id": "B"},
                {"id": "C"}
            ],

            "charges": [

                {
                    "id": "q1",
                    "value": 1.6e-19,
                    "unit": "C",
                    "at": "A"
                },

                {
                    "id": "q2",
                    "value": 1.6e-19,
                    "unit": "C",
                    "at": "B"
                },

                {
                    "id": "q3",
                    "value": 1.6e-19,
                    "unit": "C",
                    "at": "C"
                }
            ]
        },

        "geometry": [

            {
                "primitive": "EquilateralTriangle",

                "points": [
                    "A",
                    "B",
                    "C"
                ],

                "params": {

                    "side": {
                        "value": 16,
                        "unit": "cm"
                    }
                }
            }
        ],
        "relations": [],
        "constraints": [],

        "queries": [

            {
                "type": "ElectrostaticForceVector",
                "target": "q3"
            }
        ]
    }
},
{
    "problem":
        "Three charges, q1 = 8×10^-9 C, "
        "q2 = q3 = -8×10^-9 C, are placed "
        "at the three vertices of an equilateral triangle "
        "ABC with side length a = 6 cm, in air. "
        "Determine the net force acting on a charge "
        "q0 = 6×10^-9 C placed at the center O "
        "of the triangle.",

    "tags": [
        "Point",
        "EquilateralTriangle",
        "Center"
    ],

    "schema": {

        "entities": {

            "points": [

                {"id": "A"},
                {"id": "B"},
                {"id": "C"},
                {"id": "O"}
            ],

            "charges": [

                {
                    "id": "q1",
                    "value": 8e-9,
                    "unit": "C",
                    "at": "A"
                },

                {
                    "id": "q2",
                    "value": -8e-9,
                    "unit": "C",
                    "at": "B"
                },

                {
                    "id": "q3",
                    "value": -8e-9,
                    "unit": "C",
                    "at": "C"
                },

                {
                    "id": "q0",
                    "value": 6e-9,
                    "unit": "C",
                    "at": "O"
                }
            ]
        },

        "geometry": [

            {
                "primitive": "EquilateralTriangle",

                "points": [
                    "A",
                    "B",
                    "C"
                ],

                "params": {

                    "side": {
                        "value": 6,
                        "unit": "cm"
                    }
                }
            }
        ],

        "relations": [

            {
                "type": "AtCenter",

                "point": "O",

                "shape": "ABC"
            }
        ],

        "constraints": [],

        "queries": [

            {
                "type": "ElectrostaticForce",

                "target": "q0"
            }
        ]
    }
}
]

In [ ]:
import re
import json


# =========================================================
# EXPLICIT HIGH-LEVEL GEOMETRY PATTERNS
# =========================================================

EXPLICIT_PATTERNS = {
    "EquilateralTriangle": [
        r"equilateral triangle"
    ],

    "IsoscelesTriangle": [
        r"isosceles triangle"
    ],

    "RightTriangle": [
        r"right triangle",
        r"right-angled triangle"
    ],

    "Square": [
        r"\bsquare\b"
    ],

    "Rectangle": [
        r"\brectangle\b"
    ],

    "Circle": [
        r"\bcircle\b"
    ],

    "Collinear": [
        r"collinear",
        r"straight line"
    ],

    "Midpoint": [
        r"midpoint",
        r"middle point",
        r"halfway"
    ],

    "OnBisector": [
        r"perpendicular bisector"
    ]
}


# =========================================================
# RETRIEVE GEOMETRY KEYWORDS
# =========================================================

def retrieve_geometry_keywords(problem):

    text = problem.lower()

    primitives = set()
    constraints = set()

    # =====================================================
    # ALWAYS INCLUDE POINT
    # =====================================================

    primitives.add("Point")

    # =====================================================
    # EXPLICIT PATTERNS
    # =====================================================

    for keyword, patterns in EXPLICIT_PATTERNS.items():

        for pattern in patterns:

            if re.search(pattern, text):

                primitives.add(keyword)
                break

    # =====================================================
    # DISTANCE DETECTION
    # =====================================================

    if re.search(r'\b[A-Z]{2}\s*=', problem):

        constraints.add("Distance")

    if re.search(
        r'[\d.]+\s*(cm|mm|m)\s+apart',
        text
    ):

        constraints.add("Distance")

    # =====================================================
    # ANGLE DETECTION
    # =====================================================

    if re.search(
        r'angle|°|degree',
        text
    ):

        constraints.add("Angle")

    return {

        "geometry_primitives":
            sorted(primitives),

        "geometry_constraints":
            sorted(constraints)
    }
# =========================================================
# RETRIEVE_EXAMPLES
# =========================================================

def retrieve_examples(retrieval, dataset, top_k=2):

    query_tags = set(
        retrieval["geometry_primitives"] +
        retrieval["geometry_constraints"]
    )

    scored = []

    for ex in dataset:

        score = len(
            query_tags & set(ex["tags"])
        )

        scored.append((score, ex))

    scored.sort(
        key=lambda x: x[0],
        reverse=True
    )

    return [
        x[1]
        for x in scored[:top_k]
    ]

In [ ]:
# =========================================================
# BUILD FINAL PROMPT
# =========================================================

def build_schema_prompt(problem):

    retrieval = retrieve_geometry_keywords(problem)

    examples = retrieve_examples(
        retrieval,
        EXAMPLE_DATASET,
        2
    )

    prompt = f"""
You are a symbolic schema extraction engine.

Your task:
Extract a canonical symbolic schema from the problem.

IMPORTANT:
- Do NOT solve the problem.
- Output STRICT JSON ONLY.
- Preserve symbolic variables and units.
- Use ONLY retrieved geometry keywords.
- Follow schema patterns from examples.

--------------------------------------------------
RETRIEVED GEOMETRY KEYWORDS
--------------------------------------------------

{json.dumps(retrieval, indent=4)}

--------------------------------------------------
SIMILAR EXAMPLES
--------------------------------------------------
"""

    # =====================================================
    # INSERT EXAMPLES
    # =====================================================

    for i, ex in enumerate(examples, 1):

        prompt += f"""

Example {i}

Problem:
{ex["problem"]}

Schema:
{json.dumps(ex["schema"], indent=4)}
"""

    # =====================================================
    # FINAL PROBLEM
    # =====================================================

    prompt += f"""

--------------------------------------------------
OUTPUT FORMAT
--------------------------------------------------

{{
    "entities": {{}},
    "geometry": [],
    "relations": [],
    "constraints": [],
    "physics": [],
    "queries": []
}}

--------------------------------------------------
PROBLEM
--------------------------------------------------

{problem}

--------------------------------------------------
SCHEMA
--------------------------------------------------
"""

    return prompt


# =========================================================
# TEST
# =========================================================

problem = """
Three charges: q1 = +3 μC, q2 = -2 μC, and q3 = +1 μC are placed 10 cm apart on a straight line. Calculate the force acting on q2.
"""

print(build_schema_prompt(problem))


You are a symbolic schema extraction engine.

Your task:
Extract a canonical symbolic schema from the problem.

IMPORTANT:
- Do NOT solve the problem.
- Output STRICT JSON ONLY.
- Preserve symbolic variables and units.
- Use ONLY retrieved geometry keywords.
- Follow schema patterns from examples.

--------------------------------------------------
RETRIEVED GEOMETRY KEYWORDS
--------------------------------------------------

{
    "geometry_primitives": [
        "Collinear",
        "Point"
    ],
    "geometry_constraints": [
        "Distance"
    ]
}

--------------------------------------------------
SIMILAR EXAMPLES
--------------------------------------------------


Example 1

Problem:
Three charges: q1 = +3 μC, q2 = -2 μC, and q3 = +1 μC are placed 10 cm apart on a straight line. Calculate the force acting on q2.

Schema:
{
    "entities": {
        "points": [
            {
                "id": "A"
            },
            {
                "id": "B"
            },
    

In [120]:
from ollama import generate 
def extract(prompt : str):
    response = generate(
        model= "qwen2.5:7b-instruct",
        prompt=build_schema_prompt(prompt),
        options={
            'temparature':0,
            'keep_alive':'60m',
            'num_predict':4000
        }
    )
    return response['response']

print(extract("Three charges q1 = +1 μC, q2 = +1 μC, and q3 = -1 μC are placed at the vertices of an equilateral triangle with side a = 20 cm. Calculate the magnitude of the net force acting on q3."))

```json
{
    "entities": {
        "points": [
            {
                "id": "A"
            },
            {
                "id": "B"
            },
            {
                "id": "C"
            }
        ],
        "charges": [
            {
                "id": "q1",
                "value": 1,
                "unit": "μC",
                "at": "A"
            },
            {
                "id": "q2",
                "value": 1,
                "unit": "μC",
                "at": "B"
            },
            {
                "id": "q3",
                "value": -1,
                "unit": "μC",
                "at": "C"
            }
        ]
    },
    "geometry": [
        {
            "primitive": "EquilateralTriangle",
            "points": [
                "A",
                "B",
                "C"
            ],
            "params": {
                "side": {
                    "value": 20,
                    "unit": "cm"
                }
     

In [148]:
import json
import re
import math
from typing import Dict, Any, List, Tuple, FrozenSet, Optional
from collections import deque

K = 8.987551787e9  # hằng số Coulomb


# ============================================================
# 1. parse_input
# ============================================================
def parse_input(json_str: str) -> Dict[str, Any]:
    s = json_str.strip()
    # Loại bỏ markdown code fences
    match = re.search(r"```(?:json)?\s*\n(.*?)\n\s*```", s, re.DOTALL)
    if match:
        s = match.group(1).strip()
    try:
        data = json.loads(s)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON format: {e}")
    if not isinstance(data, dict):
        raise ValueError("Input must be a JSON object.")
    
    # ---- TỰ ĐỘNG SỬA TRÙNG CHARGE ID ----
    charges = data.get("entities", {}).get("charges", [])
    if charges:
        seen_ids = {}
        new_charges = []
        for c in charges:
            cid = c.get("id", "")
            if cid in seen_ids:
                seen_ids[cid] += 1
                new_id = f"{cid}_{seen_ids[cid]}"
                # Cập nhật query nếu target đến charge này
                for q in data.get("queries", []):
                    if q.get("target") == cid:
                        # Nếu đã có target trùng, ta cần sửa target thành new_id
                        # Nhưng vì đây là tự động, ta chỉ sửa charge hiện tại, 
                        # còn query sẽ trỏ đến id gốc (có thể bị sai nếu có nhiều target cùng tên)
                        # Tốt hơn là báo lỗi nếu có query target trùng id.
                        pass
                c["id"] = new_id
            else:
                seen_ids[cid] = 1
            new_charges.append(c)
        data["entities"]["charges"] = new_charges

        # Nếu có query trùng target với id gốc, cần cảnh báo hoặc sửa target phù hợp
        # Ở đây ta giả định extractor không dùng cùng id cho hai charge khác và query target rõ ràng
        # Nếu extractor trả về hai charge cùng id nhưng query chỉ target "q", ta không biết là charge nào.
        # Do đó, an toàn hơn là raise lỗi nếu phát hiện query target trùng với id bị nhân bản.
        for q in data.get("queries", []):
            target = q.get("target")
            if target in seen_ids and seen_ids[target] > 1:
                raise ValueError(
                    f"Duplicate charge id '{target}' found, and query targets this id. "
                    "Extractor must assign unique ids to each charge."
                )
    
    return data


# ============================================================
# 2. validate_input
# ============================================================
def validate_input(data: Dict[str, Any]) -> None:
    for key in ("entities", "constraints","queries"):
        if key not in data:
            raise ValueError(f"Missing '{key}' key.")

    entities = data["entities"]
    if not isinstance(entities, dict):
        raise ValueError("'entities' must be a dict.")
    if "points" not in entities or not isinstance(entities["points"], list):
        raise ValueError("'entities.points' must be a list.")
    if "charges" not in entities or not isinstance(entities["charges"], list):
        raise ValueError("'entities.charges' must be a list.")

    # Điểm
    point_ids = set()
    for p in entities["points"]:
        if "id" not in p:
            raise ValueError("Each point must have 'id'.")
        if p["id"] in point_ids:
            raise ValueError(f"Duplicate point id: {p['id']}")
        point_ids.add(p["id"])

    # Điện tích
    charge_ids = set()
    for c in entities["charges"]:
        for field in ("id", "value", "at"):
            if field not in c:
                raise ValueError(f"Charge missing '{field}'.")
        if c["unit"] == "μC":
            c["value"] = c["value"] * 10e-6
            c["unit"] = "C"
        if c["unit"] != "C":
            raise ValueError(f"Charge unit must be 'C', got {c.get('unit')}.") 
        if c["at"] not in point_ids:
            raise ValueError(f"Charge '{c['id']}' references unknown point '{c['at']}'.")
        charge_ids.add(c["id"])

    # Geometry (nếu có)
    geometry = data.get("geometry", [])
    if not isinstance(geometry, list):
        raise ValueError("'geometry' must be a list.")
    geom_covers_points = set()
    for geom in geometry:
        if not isinstance(geom, dict):
            raise ValueError("Each geometry item must be a dict.")
        if "primitive" not in geom:
            raise ValueError("Geometry item missing 'primitive'.")
        primitive = geom["primitive"]
        if primitive == "EquilateralTriangle":
            pts = geom.get("points")
            if not isinstance(pts, list) or len(pts) != 3:
                raise ValueError("EquilateralTriangle requires exactly 3 points.")
            for p in pts:
                if p not in point_ids:
                    raise ValueError(f"Geometry point '{p}' not defined.")
            params = geom.get("params")
            if not isinstance(params, dict) or "side" not in params:
                raise ValueError("EquilateralTriangle requires 'side' param.")
            side = params["side"]
            if not isinstance(side, dict) or "value" not in side or "unit" not in side:
                raise ValueError("side must have 'value' and 'unit'.")
            if side["unit"] not in ("m", "cm"):
                raise ValueError(f"side unit must be 'm' or 'cm'.")
            if float(side["value"]) <= 0:
                raise ValueError("side length must be positive.")
            geom_covers_points.update(pts)
        elif primitive == "Collinear":
            pts = geom.get("points")
            if not isinstance(pts, list) or len(pts) < 2:
                raise ValueError("Collinear requires at least 2 points.")
            for p in pts:
                if p not in point_ids:
                    raise ValueError(f"Collinear point '{p}' not defined.")
            # Collinear không yêu cầu params, nhưng nếu có thì bỏ qua
            geom_covers_points.update(pts)
        else:
            raise ValueError(f"Unsupported primitive: {primitive}")
        
    # Relations (nếu có)
    relations = data.get("relations", [])
    if not isinstance(relations, list):
        raise ValueError("'relations' must be a list.")
    for rel in relations:
        if not isinstance(rel, dict):
            raise ValueError("Each relation must be a dict.")
        rtype = rel.get("type")
        if rtype == "AtCenter":
            pt = rel.get("point")
            shape = rel.get("shape")
            if not pt or not shape:
                raise ValueError("AtCenter requires 'point' and 'shape'.")
            if pt not in point_ids:
                raise ValueError(f"AtCenter point '{pt}' not defined.")
            # shape có thể là list hoặc string
            if isinstance(shape, list):
                if len(shape) != 3:
                    raise ValueError("AtCenter shape list must have 3 points.")
                for p in shape:
                    if p not in point_ids:
                        raise ValueError(f"Shape point '{p}' not defined.")
            elif isinstance(shape, str):
                shape_pts = list(shape)
                if len(shape_pts) != 3:
                    raise ValueError(f"AtCenter shape string '{shape}' must contain exactly 3 point IDs.")
                for p in shape_pts:
                    if p not in point_ids:
                        raise ValueError(f"Shape point '{p}' not defined.")
            else:
                raise ValueError("AtCenter 'shape' must be a list or string.")
            # Bổ sung điểm tâm (đã có trong points nên không cần thêm)
        else:
            raise ValueError(f"Unsupported relation type: {rtype}")

    # Constraints
    constraints = data["constraints"]
    if not isinstance(constraints, list):
        raise ValueError("'constraints' must be a list.")
    # Chỉ bắt buộc tính liên thông nếu có ít nhất một constraint kiểu Distance
    distance_edges = []
    for con in constraints:
        if not isinstance(con, dict):
            raise ValueError("Each constraint must be a dict.")
        if con.get("type") == "Distance":
            pair = con.get("pair")
            if not isinstance(pair, list) or len(pair) != 2:
                raise ValueError("Distance constraint must have a 'pair' list of 2 point ids.")
            u, v = pair
            if u not in point_ids or v not in point_ids:
                raise ValueError(f"Distance pair ({u},{v}) uses unknown point id.")
            if u == v:
                raise ValueError("Distance pair must have distinct points.")
            unit = con.get("unit", "cm")
            if unit not in ("m", "cm"):
                raise ValueError(f"Distance unit must be 'm' or 'cm', got {unit}.")
            if float(con["value"]) <= 0:
                raise ValueError("Distance must be positive.")
            distance_edges.append((u, v))
    # Nếu có distance constraints, kiểm tra đồ thị liên thông chứa tất cả điểm
    if distance_edges:
        # BFS
        visited = set()
        start = next(iter(point_ids))
        queue = deque([start])
        visited.add(start)
        while queue:
            cur = queue.popleft()
            for u, v in distance_edges:
                if u == cur and v not in visited:
                    visited.add(v)
                    queue.append(v)
                elif v == cur and u not in visited:
                    visited.add(u)
                    queue.append(u)
        if visited != point_ids:
            missing = point_ids - visited
            raise ValueError(f"Distance graph not connected. Missing: {missing}.")
    else:
        # Nếu không có distance constraints, geometry/relations phải bao phủ được tất cả điểm
        covered_points = geom_covers_points.copy()
        # Các điểm trong relations AtCenter đã có trong point_ids, nhưng chúng được định vị nhờ shape,
        # nên shape đã có trong geometry hoặc phải được xác định bởi constraints. 
        # Ta kiểm tra xem tất cả điểm đều thuộc geometry hoặc sẽ được suy ra từ relations (tâm).
        # Trong trường hợp relations: điểm tâm phụ thuộc vào các đỉnh của shape.
        # Vì vậy, nếu các đỉnh của shape đã được geometry bao phủ, tâm cũng được coi là có thể xác định.
        for rel in data.get("relations", []):
            if rel["type"] == "AtCenter":
                shape = rel["shape"]
                if isinstance(shape, str):
                    shape_pts = list(shape)
                else:
                    shape_pts = shape
                if all(p in covered_points for p in shape_pts):
                    covered_points.add(rel["point"])
        if covered_points != point_ids:
            missing = point_ids - covered_points
            raise ValueError(f"Cannot determine coordinates for points: {missing}. Need geometry or constraints.")

    # Queries
    queries = data["queries"]
    if not isinstance(queries, list):
        raise ValueError("'queries' must be a list.")
    # Cho phép queries rỗng (không raise lỗi)
    for q in queries:
        if not isinstance(q, dict):
            raise ValueError("Each query must be a dict.")
        # qtype = q.get("type")
        # if qtype not in ("ElectrostaticForce", "ElectrostaticForceVector", "NetForceMagnitude","MagnitudeOfNetForce"):
        #     raise ValueError(f"Unsupported query type: {qtype}")
        target = q.get("target")
        if target not in charge_ids:
            raise ValueError(f"Query target '{target}' not found in charges.")


# ============================================================
# 3. solve
# ============================================================
def solve(data: Dict[str, Any]) -> Dict[str, Any]:
    validate_input(data)
    if not data["queries"]:
        return {"message": "No query to solve."}
    # Lấy điểm và điện tích
    points_list = data["entities"]["points"]
    charges_list = data["entities"]["charges"]
    charges = {c["id"]: {"value": float(c["value"]), "at": c["at"]} for c in charges_list}
    point_ids = [p["id"] for p in points_list]

    # Xây dựng tọa độ
    coords = _build_coordinates_full(data, point_ids)

    # Query
    query = data["queries"][0]  # giả sử 1 query
    target_id = query["target"]
    q_target = charges[target_id]["value"]
    target_pos = coords[charges[target_id]["at"]]

    # Tính tổng vector lực
    fx, fy = 0.0, 0.0
    for cid, cinfo in charges.items():
        if cid == target_id:
            continue
        q_other = cinfo["value"]
        other_pos = coords[cinfo["at"]]
        rx = target_pos[0] - other_pos[0]
        ry = target_pos[1] - other_pos[1]
        r = math.hypot(rx, ry)
        if r == 0:
            raise ValueError(f"Charges {target_id} and {cid} coincide.")
        F = K * q_target * q_other / (r * r)
        fx += F * rx / r
        fy += F * ry / r

    mag = math.hypot(fx, fy)
    angle = math.degrees(math.atan2(fy, fx))

    # Định dạng kết quả theo query type
    qtype = query["type"]
    if qtype == "NetForceMagnitude":
        return {"magnitude": mag, "unit": "N"}
    else:
        return {
            "force_x": fx,
            "force_y": fy,
            "magnitude": mag,
            "direction_deg": angle,
            "unit": "N"
        }


def _build_coordinates_full(data: Dict, point_ids: List[str]) -> Dict[str, Tuple[float, float]]:
    coords: Dict[str, Tuple[float, float]] = {}

    # 1. Geometry
    for geom in data.get("geometry", []):
        if geom["primitive"] == "EquilateralTriangle":
            pts = geom["points"]
            side = float(geom["params"]["side"]["value"])
            if geom["params"]["side"]["unit"] == "cm":
                side /= 100.0
            if all(p not in coords for p in pts):
                A, B, C = pts
                coords[A] = (0.0, 0.0)
                coords[B] = (side, 0.0)
                coords[C] = (side / 2.0, side * math.sqrt(3) / 2.0)
            else:
                # Nếu đã có tọa độ một phần thì cần xử lý merge – hiện chưa hỗ trợ
                raise NotImplementedError("Merging multiple geometries sharing points is not yet supported.")
        elif geom["primitive"] == "Collinear":
            _handle_collinear(geom, coords, data)
    # 2. Relations
    for rel in data.get("relations", []):
        if rel["type"] == "AtCenter":
            shape = rel["shape"]
            if isinstance(shape, str):
                shape_pts = list(shape)
            else:
                shape_pts = shape
            # Lấy tọa độ các đỉnh
            xs = [coords[p][0] for p in shape_pts]
            ys = [coords[p][1] for p in shape_pts]
            coords[rel["point"]] = (sum(xs) / 3.0, sum(ys) / 3.0)

    # 3. Constraints (Distance)
    distances: Dict[FrozenSet[str], float] = {}
    for con in data.get("constraints", []):
        if con.get("type") == "Distance":
            u, v = con["pair"]
            key = frozenset({u, v})
            val = float(con["value"])
            if con.get("unit", "cm") == "cm":
                val /= 100.0
            distances[key] = val
    if distances:
        _fill_coords_from_distances(point_ids, distances, coords)

    # Kiểm tra tất cả điểm đã có tọa độ
    for pid in point_ids:
        if pid not in coords:
            raise ValueError(f"Unable to determine coordinates for point '{pid}'.")
    return coords


def _fill_coords_from_distances(
    point_ids: List[str],
    distances: Dict[FrozenSet[str], float],
    coords: Dict[str, Tuple[float, float]]
) -> None:
    adj = {pid: {} for pid in point_ids}
    for (u, v), d in distances.items():
        # FIX: dùng trực tiếp u, v thay vì list(u), list(v)
        uid, vid = u, v
        adj[uid][vid] = d
        adj[vid][uid] = d

    placed = set(coords.keys())
    if not placed:
        root = point_ids[0]
        coords[root] = (0.0, 0.0)
        placed.add(root)
        second = None
        for p in point_ids[1:]:
            if p in adj[root]:
                second = p
                break
        if second is None:
            raise ValueError("No second point connected to root.")
        d = adj[root][second]
        coords[second] = (d, 0.0)
        placed.add(second)

    queue = deque(placed)
    while len(placed) < len(point_ids):
        found = False
        for pid in point_ids:
            if pid in placed:
                continue
            known = [n for n in adj[pid] if n in placed]
            if len(known) >= 2:
                p1, p2 = known[:2]
                x1, y1 = coords[p1]
                x2, y2 = coords[p2]
                d1 = adj[pid][p1]
                d2 = adj[pid][p2]
                dx = x2 - x1
                dy = y2 - y1
                d12 = math.hypot(dx, dy)
                if d12 == 0:
                    raise ValueError("Coincident points in distance constraints.")
                a = (d1**2 - d2**2 + d12**2) / (2 * d12)
                h_sq = d1**2 - a**2
                if h_sq < 0:
                    h_sq = 0.0
                h = math.sqrt(h_sq)
                px = x1 + a * dx / d12
                py = y1 + a * dy / d12
                rx = -dy / d12
                ry = dx / d12
                cand1 = (px + h * rx, py + h * ry)
                cand2 = (px - h * rx, py - h * ry)
                valid1 = True
                valid2 = True
                for n in known[2:]:
                    xn, yn = coords[n]
                    de = adj[pid][n]
                    if abs(math.hypot(cand1[0]-xn, cand1[1]-yn) - de) > 1e-9:
                        valid1 = False
                    if abs(math.hypot(cand2[0]-xn, cand2[1]-yn) - de) > 1e-9:
                        valid2 = False
                if valid1 and valid2:
                    coords[pid] = cand1 if cand1[1] > cand2[1] else cand2
                elif valid1:
                    coords[pid] = cand1
                elif valid2:
                    coords[pid] = cand2
                else:
                    raise ValueError(f"Ambiguous or impossible coordinates for {pid}.")
                placed.add(pid)
                queue.append(pid)
                found = True
                break
        if not found:
            unplaced = [p for p in point_ids if p not in placed]
            raise ValueError(f"Cannot place points {unplaced} with given distances.")

def _handle_collinear(geom: Dict, coords: Dict[str, Tuple[float, float]], data: Dict):
    """
    Xử lý primitive Collinear: đặt các điểm thẳng hàng trên trục x,
    tuân thủ đúng thứ tự trong danh sách 'points'.
    """
    points = geom["points"]
    # Thu thập tất cả khoảng cách từ constraints
    distances = {}
    for con in data.get("constraints", []):
        if con.get("type") == "Distance":
            u, v = con["pair"]
            key = frozenset({u, v})
            val = float(con["value"])
            if con.get("unit", "cm") == "cm":
                val /= 100.0
            distances[key] = val

    # Điểm đầu tiên làm gốc (nếu chưa có tọa độ)
    origin = points[0]
    if origin not in coords:
        coords[origin] = (0.0, 0.0)
    pos_on_line = {origin: 0.0}

    # Duyệt từng điểm tiếp theo theo thứ tự
    for i in range(1, len(points)):
        curr = points[i]
        prev_point = points[i-1]  # điểm liền trước trong danh sách

        # 1. Ưu tiên khoảng cách trực tiếp giữa curr và prev_point
        d = distances.get(frozenset({curr, prev_point}))
        if d is not None:
            pos_curr = pos_on_line[prev_point] + d
        else:
            # 2. Thử tìm khoảng cách đến một điểm bất kỳ đã biết (từ gần nhất trở về trước)
            found = False
            for j in range(i-2, -1, -1):  # duyệt ngược từ điểm trước prev_point
                known = points[j]
                d_known = distances.get(frozenset({curr, known}))
                if d_known is not None:
                    # Vì known đứng trước curr trong danh sách, khoảng cách = pos_curr - pos_known
                    pos_curr = pos_on_line[known] + d_known
                    found = True
                    break
            if not found:
                # 3. Thử tìm khoảng cách đến điểm gốc (đã thử ở trên nhưng ta làm rõ)
                d_origin = distances.get(frozenset({curr, origin}))
                if d_origin is not None:
                    pos_curr = pos_on_line[origin] + d_origin
                    found = True
            if not found:
                raise ValueError(
                    f"Cannot determine position for point '{curr}' on collinear line. "
                    "Need a distance constraint to any preceding point."
                )
        
        # Kiểm tra thứ tự: curr phải có vị trí lớn hơn điểm liền trước
        if pos_curr <= pos_on_line[prev_point]:
            raise ValueError(
                f"Collinear order violation: '{curr}' should be after '{prev_point}' "
                f"but computed position {pos_curr} <= {pos_on_line[prev_point]}."
            )

        # Lưu vị trí
        pos_on_line[curr] = pos_curr
        coords[curr] = (pos_curr, 0.0)

        # Kiểm tra tính nhất quán với tất cả các điểm đã biết khác (ngoài điểm dùng để suy)
        for p in points[:i]:
            key = frozenset({curr, p})
            if key in distances:
                actual_dist = abs(pos_curr - pos_on_line[p])
                constraint_dist = distances[key]
                if abs(actual_dist - constraint_dist) > 1e-9:
                    raise ValueError(
                        f"Inconsistent distances: {curr}-{p} constraint {constraint_dist}m, "
                        f"but positions give {actual_dist}m."
                    )

    # Kiểm tra toàn bộ các ràng buộc giữa các cặp trong collinear (phòng trường hợp thừa)
    for i in range(len(points)):
        for j in range(i+1, len(points)):
            p1, p2 = points[i], points[j]
            key = frozenset({p1, p2})
            if key in distances:
                d_constraint = distances[key]
                d_actual = abs(pos_on_line[p2] - pos_on_line[p1])
                if abs(d_actual - d_constraint) > 1e-9:
                    raise ValueError(
                        f"Constraint {p1}-{p2} = {d_constraint}m, but positions give {d_actual}m."
                    )

In [155]:
# ============================================================
# Ví dụ chạy thử
# ============================================================
if __name__ == "__main__":
    problem  = "Points A and B are separated by 20 cm in air. Charges q1 = -3 × 10^-6 C and q2 = 8 × 10^-6 C are placed at A and B, respectively. A test charge q3 = 2 × 10^-6 C is placed at point C such that AC = 12 cm and BC = 16 cm. Calculate the magnitude of the electric force acting on q3"
    data = extract(problem)
    print(data)
    data = parse_input(data)
    

```json
{
    "entities": {
        "points": [
            {
                "id": "A"
            },
            {
                "id": "B"
            },
            {
                "id": "C"
            }
        ],
        "charges": [
            {
                "id": "q1",
                "value": -3e-06,
                "unit": "C",
                "at": "A"
            },
            {
                "id": "q2",
                "value": 8e-06,
                "unit": "C",
                "at": "B"
            },
            {
                "id": "q3",
                "value": 2e-06,
                "unit": "C",
                "at": "C"
            }
        ]
    },
    "geometry": [],
    "constraints": [
        {
            "type": "Distance",
            "pair": [
                "A",
                "B"
            ],
            "value": 20,
            "unit": "cm"
        },
        {
            "type": "Distance",
            "pair": [
                "C",


In [156]:
result = solve(data)
print("Electrostatic force on q3:")
print(result)

Electrostatic force on q3:
{'force_x': -6.740663840250001, 'force_y': 0.37448132445833204, 'magnitude': 6.751058085190939, 'direction_deg': 176.82016988013578, 'unit': 'N'}
